In [1]:
import random

def generar_datos_legacy(nombre_archivo="envios_legacy.txt", cantidad_registros=500000):
    """
    Genera un archivo de texto simulando un volcado de base de datos antiguo.
    Incluye ruido y errores intencionales para probar pipelines de datos.
    """
    estados = ["ENTREGADO", "EN_RUTA", "CANCELADO", "entregado", " cancelado ", " EN_RUTA"]
    zonas = ["Norte", "Sur", "Este", "Oeste", "Centro", "norte", " SUR "]

    print(f"[*] Generando {cantidad_registros} registros de prueba...")

    with open(nombre_archivo, "w", encoding="utf-8") as f:
        f.write("ID_PAQUETE| ESTADO | DISTANCIA_KM | COSTO_USD | ZONA\n")
        
        for i in range(1, cantidad_registros + 1):
            estado = random.choice(estados)
            distancia = round(random.uniform(2.0, 150.0), 1)
            costo = random.randint(5, 800)
            zona = random.choice(zonas)
            
           
            if i % 1000 == 0:
                f.write(f"PKG{i}|FATAL_ERROR_SISTEMA_CAIDO_LINEA_INCOMPLETA\n")

            elif i % 1500 == 0:
                f.write(f" PKG{i} | {estado} | {distancia} | NULL_VALUE | {zona} \n")

            else:
                f.write(f" PKG{i} | {estado} |  {distancia} | {costo} | {zona} \n")

    print(f"[+] ¡Éxito! Archivo '{nombre_archivo}' creado. Ya podés correr tu ETL.")

generar_datos_legacy()

[*] Generando 500000 registros de prueba...


[+] ¡Éxito! Archivo 'envios_legacy.txt' creado. Ya podés correr tu ETL.


In [2]:
import requests
import pandas as pd
from IPython.display import display

def reintentar_api(func):
    def wrapper(*args, **kwargs):
        for i in range(3):
            try:
                return func(*args, **kwargs)
            except Exception:
                print("error, reintentando")
        return None
    return wrapper

@reintentar_api
def obtener_dolar():
    r = requests.get("https://dolarapi.com/v1/dolares/oficial")
    data = r.json()
    return data["venta"]

def leer_archivo(ruta):
    with open(ruta, "r") as f:
        next(f)

        for linea in f:
            try:
                partes = linea.split("|")

                if len(partes) != 5:
                    continue

                id = partes[0].strip()
                estado = partes[1].strip().upper()
                distancia = float(partes[2].strip())
                costo = partes[3].strip()
                zona = partes[4].strip().upper()

                if costo == "NULL_VALUE":
                    continue

                costo = float(costo)

                yield {
                    "id": id,
                    "estado": estado,
                    "distancia": distancia,
                    "costo": costo,
                    "zona": zona
                }

            except Exception:
                continue

def transformar(datos):

    dolar = obtener_dolar()

    filtrados = filter(
        lambda x: x["estado"] != "CANCELADO" and x["distancia"] >= 20,
        datos
    )

    transformados = map(
        lambda x: {
            "id": x["id"],
            "zona": x["zona"],
            "costo_final": x["costo"] * dolar * 1.21
        },
        filtrados
    )

    ordenados = sorted(
        transformados,
        key=lambda x: x["costo_final"],
        reverse=True
    )

    return ordenados

def guardar(datos, ruta):
    df = pd.DataFrame(datos)

    df["costo_final"] = df["costo_final"].round(2)

    display(df)

    df.to_csv(ruta, index=False, sep=";")

def main():
    datos = leer_archivo("envios_legacy.txt")
    procesados = transformar(datos)
    guardar(procesados, "reporte_logistica_limpio.csv")
    print("Proceso terminado")

main()

,id,zona,costo_final
0,PKG335,NORTE,1379400.00
1,PKG2768,NORTE,1379400.00
2,PKG3221,SUR,1379400.00
3,PKG3598,OESTE,1379400.00
4,PKG4047,SUR,1379400.00
...,...,...,...
292773,PKG494310,NORTE,8621.25
292774,PKG497052,OESTE,8621.25
292775,PKG497918,NORTE,8621.25
292776,PKG498302,CENTRO,8621.25


Proceso terminado
